# Train spatial/non-spatial classifier

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
SEED = 42
np.random.seed(SEED)

## ✂ Split our manually labelled dataset into train/val/test (80/10/10)

In [3]:
data = (
    pd.read_excel('interim/is_geospatial_20260428_edits.xlsx')
    .dropna(subset=['is_geospatial_human_check'])
    .assign(
        y=lambda df_: df_.is_geospatial_human_check.astype(int)
    )
    .filter(['query_id', 'query_text', 'y'])
)

print(data.y.value_counts())

data

y
0    632
1    568
Name: count, dtype: int64


,query_id,query_text,y
0,891609,what river runs through southern california,1
1,848519,what is the statue of limitation to collect a ...,0
2,485574,reactive arthritis from salmonella,0
3,375479,how to register customary land in png,1
4,314185,how much does insulin cost for humans,0
...,...,...,...
4976,1152306,what is the adr,0
4981,848484,what is the state tree for washington,1
4983,945399,when do you renew your massachusetts driver's ...,1
4986,543451,"weather in beaumont, california fahrenheit",1


In [15]:
# First split: hold out 800 for test
train_val_df, test_df = train_test_split(
    data,
    test_size=800,
    random_state=SEED,
    stratify=data['y'] # ensure equal number of spatial/non-spatial
)

# Second split: from the remaining 400, take 200 for val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=200,
    random_state=SEED,
    stratify=train_val_df['y']
)

In [16]:
assert len(train_df) + len(val_df) + len(test_df) == len(data)

print('==Train counts==')
print(train_df.y.value_counts())

print('\n\n==Val counts==')
print(val_df.y.value_counts())

print('\n\n==Test counts==')
print(test_df.y.value_counts())

==Train counts==
y
0    105
1     95
Name: count, dtype: int64


==Val counts==
y
0    106
1     94
Name: count, dtype: int64


==Test counts==
y
0    421
1    379
Name: count, dtype: int64


In [17]:
train_df.to_csv('output/is_geospatial.train.csv', index=False)
val_df.to_csv('output/is_geospatial.val.csv', index=False)
test_df.to_csv('output/is_geospatial.test.csv', index=False)

### Inter-annotator agreement

- Cohen's kappa: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html

In [78]:
from sklearn.metrics import cohen_kappa_score

In [86]:
iaa_ilya = pd.read_csv('output/iaa_ilya.csv').set_index('query_id').rename(columns={'y': 'ilya'})
iaa_stefano = pd.read_csv('output/iaa_stefano.csv').set_index('query_id').rename(columns={'y': 'stefano'})
iaa_james = pd.read_csv('output/iaa_james.csv').set_index('query_id').rename(columns={'y': 'james'})

In [99]:
(
    iaa_ilya.merge(
        iaa_stefano, left_index=True, right_index=True
    ).merge(
        iaa_james, left_index=True, right_index=True
    #).query('ilya != james or ilya != stefano or james != stefano')
    ).query('ilya == james and ilya == stefano and james == stefano')
    .filter(['ilya', 'james', 'stefano', 'query_text'])
    .ilya.value_counts()
)

ilya
0    107
1     72
Name: count, dtype: int64

In [94]:
kappa_ilya_stefano = cohen_kappa_score(iaa_ilya.ilya, iaa_stefano.stefano)
kappa_ilya_james = cohen_kappa_score(iaa_ilya.ilya, iaa_james.james)
kappa_james_stefano = cohen_kappa_score(iaa_james.james, iaa_stefano.stefano)

print('Ilya--Stefano: ', round(kappa_ilya_stefano,2))
print('Ilya--James: ', round(kappa_ilya_james, 2))
print('James--Stefano: ', round(kappa_james_stefano, 2))

Ilya--Stefano:  0.88
Ilya--James:  0.85
James--Stefano:  0.84


# 💪 Train our classifier

In [17]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
import torch

In [18]:
train_ds = Dataset.from_pandas(
    pd.read_csv('output/is_geospatial.train.csv', usecols=['query_text', 'y'])
)

val_ds = Dataset.from_pandas(
    pd.read_csv('output/is_geospatial.val.csv', usecols=['query_text', 'y'])
)

test_ds = Dataset.from_pandas(
    pd.read_csv('output/is_geospatial.test.csv', usecols=['query_text', 'y'])
)

In [19]:
model = SetFitModel.from_pretrained('BAAI/bge-small-en-v1.5')
# warning of initialising classification head with random weights is OK and expected!

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [20]:
args = TrainingArguments(
    num_epochs=5,
    num_iterations=20,
    batch_size=64,
    body_learning_rate=2e-5,
    seed=SEED,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='embedding_loss',
    greater_is_better=False
)

def compute_metrics(y_pred, y_true):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    column_mapping={'query_text': 'text', 'y': 'label'},
    metric=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# disable pin_memory in the internal HF trainer
trainer.st_trainer.args.dataloader_pin_memory = False

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [21]:
trainer.train()

***** Running training *****
  Num unique pairs = 8000
  Batch size = 64
  Num epochs = 5


Epoch,Training Loss,Validation Loss
1,0.061400,0.064691
2,0.001400,0.066151
3,0.001100,0.064633
4,0.000900,0.065272
5,0.000900,0.065276


In [22]:
results = trainer.evaluate(test_ds)
print(f"Accuracy: {results['accuracy']:.3f}")
print(f"F1: {results['f1']:.3f}")

Applying column mapping to the evaluation dataset
***** Running evaluation *****


Accuracy: 0.934
F1: 0.930


In [23]:
test_texts = test_df['query_text'].tolist()
test_labels = test_df['y'].tolist()
preds = model.predict(test_texts)

# Build a results dataframe
results_df = pd.DataFrame({
    'text': test_texts,
    'true': test_labels,
    'pred': preds,
})

# Convert preds to int if they come back as tensors/strings
results_df['pred'] = results_df['pred'].astype(int)
results_df['true'] = results_df['true'].astype(int)

# Filter to errors
errors = results_df[results_df['true'] != results_df['pred']].copy()
errors['error_type'] = errors.apply(
    lambda r: 'false_positive' if r['pred'] == 1 else 'false_negative',
    axis=1,
)

print(f'Total errors: {len(errors)} / {len(results_df)}')
print(errors['error_type'].value_counts())

Total errors: 53 / 800
error_type
false_negative    27
false_positive    26
Name: count, dtype: int64


In [24]:
# False positives - model said 1, truth is 0
fp = errors[errors['error_type'] == 'false_positive']
print(f'\n=== False positives ({len(fp)}) ===')
for _, row in fp.head(20).iterrows():
    print(f'- {row["text"]}')

# False negatives - model said 0, truth is 1
fn = errors[errors['error_type'] == 'false_negative']
print(f'\n=== False negatives ({len(fn)}) ===')
for _, row in fn.head(20).iterrows():
    print(f'- {row["text"]}')


=== False positives (26) ===
- when is the good time to see northern lights
- what is microsoft al
- est uber cost
- what is the length of earth days on saturn
- state license board number for contractors
- what is bardo
- what is the name of the us armed forces telephone network
- what conference is it notre dame basketball on
- what all places do i need to change my address when i move
- when are the peak seasons prices at disneyland
- where does cantonese food come from
- what time is mid shift at southern wine and spirits
- biggest snakes in world
- what is in dcd
- when did the gulf of tonkin incident happen
- how many teams are in the eastern conference
- where is area code
- where is the minecraft spawner in pc
- what are those who live in the desert called
- where does the name isla originate from

=== False negatives (27) ===
- how much do granite fabricators pay for granite
- va tax contact
- how many steps in the arc de triomphe
- who's statue is at the top of little round 

In [33]:
trainer.model.save_pretrained('geospatial_query_classifier_eval')

### Let's stress-test it (sanity/sense check)

In [34]:
model = SetFitModel.from_pretrained('geospatial_query_classifier_eval')

In [35]:
def is_spatial(q):
    pred = model.predict([q])
    print(f'{q}: {"✅" if pred[0] else "🚫"} \n---')

In [36]:
is_spatial('is it a long flight london to kaunas')

is it a long flight london to kaunas: ✅ 
---


##### clearly spatial
is_spatial('cafes within walking distance')
is_spatial('nearest hospital')
is_spatial('distance from paris to berlin')
is_spatial('where am i right now')
is_spatial('countries bordering ukraine')
is_spatial('borderline invisible')
is_spatial('elevation of mount fuji')
is_spatial('how far is the nearest bus stop')
is_spatial('route from oxford to cambridge')
is_spatial('restaurants along the m1')
is_spatial('stop it m8')
is_spatial('flood risk in this area')

# clearly non-spatial (metaphorical / abstract)
is_spatial('close to my heart')
is_spatial('far from the truth')
is_spatial('a long way to go in my career')
is_spatial('high-level overview')
is_spatial('deep learning basics')
is_spatial('near impossible')
is_spatial('where do i stand politically')
is_spatial('on the edge emotionally')

# ambiguous / borderline (good test cases)
is_spatial('where should i live')              # needs disambiguation
is_spatial('how far can i go with this idea')  # metaphorical by default
is_spatial('where should i invest my money')   # abstract 'where'
is_spatial('how close are we to a solution')
is_spatial('what is my position on this')
is_spatial('where does this leave us')
is_spatial('how far apart are the classes')

# mixed spatial + non-spatial intent
is_spatial('how far is too far in relationships')
is_spatial('close friends who live far away')
is_spatial('where can i escape mentally')
is_spatial('distance learning programmes near me')

# tricky linguistic traps
is_spatial('where do i belong')
is_spatial('long way home')
is_spatial('keep your distance')
is_spatial('at a crossroads in life')
is_spatial('moving forward with the plan')

## Let's bootstrap to calculate confidence intervals for accuracy and F1 scores

In [38]:
y_true = np.asarray(test_labels)
y_pred = np.asarray(preds)

rng = np.random.default_rng(SEED)
n = len(y_true)
n_boot = 1000

acc_boot, f1_boot = [], []
for _ in range(n_boot):
    idx = rng.integers(0, n, size=n)  # sample with replacement
    acc_boot.append(accuracy_score(y_true[idx], y_pred[idx]))
    f1_boot.append(f1_score(y_true[idx], y_pred[idx]))

acc_ci = np.percentile(acc_boot, [2.5, 97.5])
f1_ci  = np.percentile(f1_boot,  [2.5, 97.5])

In [39]:
print('Accuracy CI 95%:', acc_ci)
print('F1 CI 95%:', f1_ci)

Accuracy CI 95%: [0.915 0.95 ]
F1 CI 95%: [0.90885045 0.94737498]


## Let's now train on the whole labelled set (1,200)

- We have measured F1 and accuracy above; now let's retrain on the whole train+val+test set
- We would expect f1 to increase slightly because of more data 

In [52]:
from datasets import concatenate_datasets
from setfit import SetFitModel, Trainer, TrainingArguments
from huggingface_hub import upload_folder

In [45]:
full_ds = concatenate_datasets([train_ds, val_ds, test_ds])

In [46]:
pd.Series(full_ds['y']).value_counts()

0    632
1    568
Name: count, dtype: int64

In [49]:
model_prod = SetFitModel.from_pretrained('BAAI/bge-small-en-v1.5')

args_prod = TrainingArguments(
    num_epochs=3, # above best result after 3 epochs (lowest validation loss) so use 3 not 5
    num_iterations=20,
    batch_size=64,
    body_learning_rate=2e-5,
    seed=SEED,
    logging_steps=100,
    eval_strategy='no',
    save_strategy='no',
    load_best_model_at_end=False,
)

trainer_prod = Trainer(
    model=model_prod,
    args=args_prod, # using same args as above except for eval strategy
    train_dataset=full_ds,
    eval_dataset=None,  # no validation - we are not tuning this time!
    column_mapping={'query_text': 'text', 'y': 'label'},
)

trainer_prod.st_trainer.args.dataloader_pin_memory = False
trainer_prod.train()

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 48000
  Batch size = 64
  Num epochs = 3


Step,Training Loss
1,0.237400
100,0.226600
200,0.066100
300,0.008000
400,0.002700
500,0.001300
600,0.001100
700,0.000900
800,0.000700
900,0.000700


In [53]:
# Save locally (just in case)
trainer_prod.model.save_pretrained('is-geospatial-query')

# Push directly to the Huggingface Hub
# ONLY ONCE -- otherwise it will overwrite README
#trainer_prod.model.push_to_hub(
#    'ilyankou/is-geospatial-query',
#    commit_message='Upload v1.0',
#    private=False
#)

# Push to HuggingFace - everything BUT README
upload_folder(
    folder_path='./is-geospatial-query',
    repo_id='ilyankou/is-geospatial-query',
    commit_message='Upload v1.1',
    ignore_patterns=['README.md']
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...-geospatial-query/model.safetensors:   6%|5         | 8.00MB /  133MB            

  .../is-geospatial-query/model_head.pkl:   1%|          |  37.0B / 3.94kB            

CommitInfo(commit_url='https://huggingface.co/ilyankou/is-geospatial-query/commit/5e6106bf3eb48038ee4eb89d17c1eb4e538a5a26', commit_message='Upload v1.1', commit_description='', oid='5e6106bf3eb48038ee4eb89d17c1eb4e538a5a26', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ilyankou/is-geospatial-query', endpoint='https://huggingface.co', repo_type='model', repo_id='ilyankou/is-geospatial-query'), pr_revision=None, pr_num=None)

# Classify *all* 1.1M MS MARCO queries

- Runs about 10 min (on an M3 processor)

In [54]:
from setfit import SetFitModel
import pandas as pd
from tqdm import tqdm

In [55]:
# Load the model
spatial_classifier = SetFitModel.from_pretrained("ilyankou/is-geospatial-query")

# Read queries
queries = pd.read_csv("interim/queries.csv.zip")

# Define batch prediction function
def batch_predict(texts, batch_size=1024):
    preds = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        preds.extend(p.item() for p in spatial_classifier.predict(batch))
    return preds #.item()

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

config_setfit.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model_head.pkl:   0%|          | 0.00/3.94k [00:00<?, ?B/s]

In [56]:
def ask_geospatial_detector(q):
    pred = spatial_classifier.predict([q])
    print(f'{q}: {"✅" if pred[0] else "🚫"} \n---')

ask_geospatial_detector('countries bordering ukraine')
ask_geospatial_detector('elevation of mount fuji')
ask_geospatial_detector('who when and where')
ask_geospatial_detector('where are they hiding')
ask_geospatial_detector("nearest hospital")
ask_geospatial_detector("far from the truth")
ask_geospatial_detector("close to my heart")
ask_geospatial_detector("flood risk in this area")

countries bordering ukraine: ✅ 
---
elevation of mount fuji: ✅ 
---
who when and where: 🚫 
---
where are they hiding: ✅ 
---
nearest hospital: ✅ 
---
far from the truth: 🚫 
---
close to my heart: 🚫 
---
flood risk in this area: ✅ 
---


In [57]:
# Run prediction
queries['is_geospatial_pred'] = batch_predict( queries['query_text'].tolist() )

100%|█████████████████████████████████████████| 988/988 [11:20<00:00,  1.45it/s]


In [59]:
queries.groupby('is_geospatial_pred', group_keys=False).sample(n=5, random_state=SEED)

,query_id,query_text,is_geospatial_pred
647794,859573,what is urine leukocyte esterase positive,0
863348,956194,when to use long term urinary catheters,0
809679,867242,what kind of dance shakira does,0
52613,386533,how to write a children's picture book,0
71014,249668,how long does a laptop battery work,0
1007521,986099,where is lagos city,1
419907,845090,what is the russian annexation of crimea?,1
375956,606603,"what county is grand island, ne",1
859577,999233,where is union ohio,1
493617,1171704,populations of greater tehachapi ca,1


In [60]:
queries.to_csv('output/queries-classified.20260501.csv.zip', index=False)

## Let's look at classification stats

In [61]:
final = pd.read_csv('output/queries-classified.20260501.csv.zip')
final

,query_id,query_text,is_geospatial_pred
0,1048578,cost of endless pools/swim spa,0
1,1048579,what is pcnt,0
2,1048580,what is pcb waste,0
3,1048581,what is pbis?,0
4,1048582,what is paysky,0
...,...,...,...
1010911,633855,what does canada post regulations mean,1
1010912,1059728,wholesale lularoe price,0
1010913,210839,how can i watch the day after,0
1010914,908165,what to use instead of pgp in windows,0


In [62]:
final.is_geospatial_pred.value_counts()

is_geospatial_pred
0    829089
1    181827
Name: count, dtype: int64

In [71]:
(final.is_geospatial_pred.eq(1).mean() * 100).round(1)

18.0

In [64]:
# What is the most common first word in geospatial queries?
top_20_spatial = (final[final.is_geospatial_pred.eq(1)].query_text.str.split(' ').str[0].str.lower().value_counts()
    / final.is_geospatial_pred.eq(1).sum()
     * 100
).head(20).round(1)

top_20_spatial

query_text
what           29.6
where          15.8
how            11.6
average         3.4
when            3.3
weather         2.8
is              2.5
which           1.9
who             1.8
why             1.1
cost            1.0
population      0.9
what's          0.9
does            0.6
can             0.5
most            0.5
largest         0.5
distance        0.5
temperature     0.5
the             0.4
Name: count, dtype: float64

In [65]:
# What about non-geospatial queries?
top_20_nonspatial = (final[final.is_geospatial_pred.eq(0)].query_text.str.split(' ').str[0].str.lower().value_counts()
     / final.is_geospatial_pred.eq(0).sum()
     * 100
).head(20).round(1)

top_20_nonspatial

query_text
what          36.1
how           17.9
who            3.7
is             3.0
when           2.6
can            2.1
why            1.8
which          1.8
does           1.3
average        1.2
define         1.0
definition     1.0
cost           0.9
where          0.8
do             0.7
the            0.6
are            0.6
meaning        0.5
what's         0.4
causes         0.4
Name: count, dtype: float64

In [66]:
pd.concat([top_20_spatial, top_20_nonspatial]).index.value_counts()

query_text
what           2
who            2
the            2
can            2
where          2
what's         2
cost           2
why            2
does           2
which          2
how            2
when           2
average        2
is             2
meaning        1
are            1
do             1
definition     1
define         1
distance       1
temperature    1
largest        1
most           1
weather        1
population     1
causes         1
Name: count, dtype: int64